In [1]:
import argparse
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
matplotlib.use('Agg')
import joblib

from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor


In [2]:
# Configuration
# --------------------------------------------------------------------------- #

RAW_FEATURES = [
    "irradiance_wm2",
    "rainfall_mm",
    "relative_humidity_pct",
    "sea_level_pressure_hpa",
    "temperature_c",
    "visibility_km",
    "wind_speed_ms",
]

TIME_FEATURES = ["hour", "minute", "day", "month", "year"]

# Cyclical encodings derived from TIME_FEATURES (see feature engineering step)
CYCLICAL_FEATURES = ["hour_sin", "hour_cos", "month_sin", "month_cos"]

# Station/site identifier — critical for capturing per-site capacity/calibration
SITE_FEATURE = "site_id_ttl"

ALL_FEATURES = RAW_FEATURES + CYCLICAL_FEATURES + ["day"] + [SITE_FEATURE]

TARGET_COL = "normalized_generation"
FUTURE_TARGET_COL = "target_1h"

# 15-minute native sampling interval; 1 hour ahead = 4 steps.
INTERVAL = pd.Timedelta(minutes=15)
FORECAST_STEPS = 4
FORECAST_DELTA = INTERVAL * FORECAST_STEPS  # exactly 1 hour

TRAIN_FRACTION = 0.8

OUTPUT_GRAPH_DIR = "output_graph"
MODELS_DIR = "models"
METRICS_DIR = "metrics"


In [3]:
# 1. Load
df = pd.read_csv('data/stage-3/training_data.csv')
test_df = pd.read_csv('data/stage-3/test_data.csv')

print(df.columns.tolist())
print(test_df.columns.tolist())


['hour', 'minute', 'day', 'month', 'year', 'irradiance_wm2', 'rainfall_mm', 'relative_humidity_pct', 'sea_level_pressure_hpa', 'temperature_c', 'visibility_km', 'wind_speed_ms', 'normalized_generation']
['hour', 'minute', 'day', 'month', 'year', 'irradiance_wm2', 'rainfall_mm', 'relative_humidity_pct', 'sea_level_pressure_hpa', 'temperature_c', 'visibility_km', 'wind_speed_ms', 'normalized_generation']


In [4]:
for d in [df, test_df]:
    d['hour_sin'] = np.sin(2 * np.pi * d['hour'] / 24)
    d['hour_cos'] = np.cos(2 * np.pi * d['hour'] / 24)
    d['month_sin'] = np.sin(2 * np.pi * d['month'] / 12)
    d['month_cos'] = np.cos(2 * np.pi * d['month'] / 12)


In [5]:

df.corr(numeric_only = True)

,hour,minute,day,month,year,irradiance_wm2,rainfall_mm,relative_humidity_pct,sea_level_pressure_hpa,temperature_c,visibility_km,wind_speed_ms,normalized_generation,hour_sin,hour_cos,month_sin,month_cos
hour,1.000000,-0.000250,0.000656,-0.001380,0.000489,0.029521,-0.011458,-0.091171,-0.027346,0.065428,0.061804,-0.022726,0.030210,-0.775819,-0.101566,0.000392,0.000172
minute,-0.000250,1.000000,-0.000183,-0.000320,-0.000038,0.003075,-0.001533,-0.000394,0.000175,0.000442,0.000174,0.000826,-0.000609,0.000484,0.000277,0.000070,-0.000297
day,0.000656,-0.000183,1.000000,0.017137,-0.000522,-0.021559,-0.002733,0.042778,0.035479,-0.003745,-0.065395,-0.093167,-0.019900,-0.000788,0.000095,-0.008754,0.003428
month,-0.001380,-0.000320,0.017137,1.000000,-0.376029,0.037218,-0.013659,-0.281297,0.017274,0.158320,0.160366,0.190947,0.010964,0.001862,0.000353,-0.696647,0.371876
year,0.000489,-0.000038,-0.000522,-0.376029,1.000000,0.010142,-0.013308,0.005964,0.168620,-0.199224,-0.121048,0.003679,-0.009338,-0.000307,-0.000415,0.387508,-0.006030
irradiance_wm2,0.029521,0.003075,-0.021559,0.037218,0.010142,1.000000,-0.046301,-0.492849,-0.081509,0.416717,0.200022,0.042306,0.952947,0.027299,-0.693959,-0.079130,-0.057782
rainfall_mm,-0.011458,-0.001533,-0.002733,-0.013659,-0.013308,-0.046301,1.000000,0.096854,-0.064451,-0.001956,-0.301009,0.040781,-0.046360,0.012781,-0.022642,-0.009124,-0.057611
relative_humidity_pct,-0.091171,-0.000394,0.042778,-0.281297,0.005964,-0.492849,0.096854,1.000000,-0.239741,-0.054089,-0.401613,-0.193678,-0.481652,0.088314,0.332305,0.088187,-0.324034
sea_level_pressure_hpa,-0.027346,0.000175,0.035479,0.017274,0.168620,-0.081509,-0.064451,-0.239741,1.000000,-0.797375,-0.005677,0.169385,-0.092367,0.077998,0.042033,0.411470,0.720286
temperature_c,0.065428,0.000442,-0.003745,0.158320,-0.199224,0.416717,-0.001956,-0.054089,-0.797375,1.000000,0.179165,-0.255496,0.423926,-0.066954,-0.241606,-0.569793,-0.595761


In [6]:
features = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day',
    'irradiance_wm2', 'rainfall_mm', 'relative_humidity_pct',
    'sea_level_pressure_hpa', 'temperature_c', 'visibility_km', 'wind_speed_ms',
]
target = 'normalized_generation'

# Ensure datetime exists on both frames before splitting
df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour', 'minute']])
test_df['datetime'] = pd.to_datetime(test_df[['year', 'month', 'day', 'hour', 'minute']])

X_train, y_train = df[features], df[target]
X_test, y_test = test_df[features], test_df[target]

# Keep datetime aligned with X_test/y_test by index, for plotting later
dt_test = test_df['datetime']

# Use last ~3 months of 2022 as validation, rest as training
val_mask = (df['year'] == 2022) & (df['month'] >= 10)

X_tr, y_tr = df.loc[~val_mask, features], df.loc[~val_mask, target]
X_val, y_val = df.loc[val_mask, features], df.loc[val_mask, target]

# Keep datetime aligned with train/val splits too, for any future plotting
dt_tr = df.loc[~val_mask, 'datetime']
dt_val = df.loc[val_mask, 'datetime']

In [7]:

features = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day',
    'irradiance_wm2', 'rainfall_mm', 'relative_humidity_pct',
    'sea_level_pressure_hpa', 'temperature_c', 'visibility_km', 'wind_speed_ms',
]
target = 'normalized_generation'

X_tr, y_tr = df.loc[~val_mask, features], df.loc[~val_mask, target]
X_val, y_val = df.loc[val_mask, features], df.loc[val_mask, target]
X_test, y_test = test_df[features], test_df[target]

# --- Main model ---
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1  # Use all available cores
)

model.fit(X_tr, y_tr)

RandomForestRegressor(max_depth=15, n_jobs=-1, random_state=42)

In [8]:
y_pred = model.predict(X_test)

print("RMSE:", root_mean_squared_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R²:", r2_score(y_test, y_pred))


RMSE: 0.027284585840071585
MSE: 0.0007444486244642349
MAE: 0.015539686627424163
R²: 0.7392168075783384


In [9]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

y_pred3 = model.predict(X_test)

# Pick a random day from the available dates in test_df
available_days = test_df['datetime'].dt.date.unique()
random_day = np.random.choice(available_days)
random_day = pd.Timestamp(random_day)

mask = (test_df['datetime'] >= random_day) & (test_df['datetime'] < random_day + pd.Timedelta(days=2))
times = test_df.loc[mask, 'datetime']

actual_vals = y_test[mask.values]
pred_vals = y_pred3[mask.values]

peak_actual = actual_vals.max()
peak_pred = pred_vals.max()

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(times, actual_vals, color='black', linewidth=1.5, label='Actual')
ax.plot(times, pred_vals, color='#2ecc71', linewidth=1.5, label='Random Forest Model')

ax.set_title('Actual vs Random Forest Model — All Sites', fontsize=13)
ax.set_xlabel('Time')
ax.set_ylabel('normalized_generation')
ax.legend(loc='upper right', frameon=True)
ax.grid(alpha=0.3)

# Date + peak power level label
label_text = (
    f"{random_day.strftime('%B %d, %Y')}\n"
    f"Peak Actual: {peak_actual:.3f} kwh\n"
    f"Peak Predicted: {peak_pred:.3f} kwh"
)

ax.text(
    0.02, 0.95, label_text,
    transform=ax.transAxes,
    fontsize=12, fontweight='bold',
    verticalalignment='top',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.9)
)

plt.tight_layout()
plt.savefig('actual_vs_rf_random_day.png', dpi=150, bbox_inches='tight', facecolor='white')
print(f"Plotted random day: {random_day.strftime('%Y-%m-%d')}")
print("Saved to actual_vs_rf_random_day.png")


Plotted random day: 2023-10-15
Saved to actual_vs_rf_random_day.png


In [10]:
import joblib

# Save model using joblib
joblib.dump(model, "urja-rf.joblib")

['urja-rf.joblib']